# Final French evaluation on Kaggle GPU

Enable **GPU** and **Internet**. Add the private `constitution-evidence-final-input.zip` bundle as a Kaggle input. This recovery bundle contains the previous T4 result ZIP: it preserves the 90 BM25 and 90 hybrid rows and runs only the 90 full-text questions by filling an offloaded attention cache in 512-token chunks. Run the cells in order. The first install may require a kernel restart; then run all cells again. Keep the input dataset and downloaded results private.


In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys, zipfile
INPUT = Path('/kaggle/input')
ROOT = Path('/kaggle/working/moroccan-constitution-rag')
archives = sorted(INPUT.rglob('constitution-evidence-final-input.zip'))
folders = sorted({p.parent.parent for p in INPUT.rglob('kaggle-bundle.json') if p.parent.name == 'runs'})
if len(archives) > 1 or (not archives and len(folders) != 1):
    raise RuntimeError(f'Expected one private final input; archives={archives}, extracted folders={folders}')
if archives:
    with zipfile.ZipFile(archives[0]) as bundle:
        incoming = json.loads(bundle.read('runs/kaggle-bundle.json'))
else:
    incoming = json.loads((folders[0]/'runs/kaggle-bundle.json').read_text())
if ROOT.exists():
    existing_path = ROOT/'runs/kaggle-bundle.json'
    existing = json.loads(existing_path.read_text()) if existing_path.exists() else {}
    if existing.get('commit') != incoming['commit']:
        if (ROOT/'runs/final-reserved-qwen4b-transformers').exists():
            raise RuntimeError('Existing final results use a different input commit; save them before changing inputs')
        shutil.rmtree(ROOT)
if archives:
    with zipfile.ZipFile(archives[0]) as bundle:
        for name in bundle.namelist():
            path = Path(name)
            if path.is_absolute() or '..' in path.parts:
                raise RuntimeError('Unsafe archive path')
        ROOT.mkdir(parents=True, exist_ok=True)
        bundle.extractall(ROOT)
else:
    shutil.copytree(folders[0], ROOT, dirs_exist_ok=True)
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT/'src'))
from scripts.kaggle_bundle_loader import restore_git_metadata
print('Bundle commit:', restore_git_metadata(ROOT))
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout.strip())


In [ ]:
import hashlib, importlib, os
requirements = ['-e','.[dense]','transformers==4.57.6','accelerate>=1,<2',
                'bitsandbytes>=0.46,<1','huggingface_hub>=0.35,<1']
signature = hashlib.sha256(json.dumps(requirements).encode()).hexdigest()
marker = Path('/kaggle/working/constitution-final-env.txt')
if not marker.exists() or marker.read_text().strip() != signature:
    subprocess.run([sys.executable,'-m','pip','install','-q',*requirements],check=True,cwd=ROOT)
    marker.write_text(signature+'\n')
    raise RuntimeError('Packages installed. Restart the Kaggle kernel, then rerun all cells.')
importlib.invalidate_caches()
import torch, transformers, bitsandbytes, fastembed, faiss
if not torch.cuda.is_available():
    raise RuntimeError('Select a Kaggle GPU accelerator')
print('Torch',torch.__version__,'Transformers',transformers.__version__,'GPU',torch.cuda.get_device_name(0))


In [ ]:
from scripts.kaggle_final_runtime import verify_bundle
from constitution_evidence.release import release_readiness
result = verify_bundle(ROOT)
readiness = release_readiness(ROOT/'runs/complete-owner-reviewed-benchmark.jsonl',
    ROOT/'data/processed/manifest.json', ROOT/'runs/validation-selection.json',
    ROOT/'configs/final_systems.json', repo_root=ROOT)
if readiness['status'] != 'ready':
    raise RuntimeError(f"Release gate blocked: {readiness['reasons']}")
print('Ready at commit',result['commit'],'with',len(result['files']),'verified private inputs')


In [ ]:
from scripts.kaggle_final_runtime import execute
# This saves each answer row immediately and resumes the same run after interruption.
summary = execute(ROOT)
print(json.dumps(summary,ensure_ascii=False,indent=2))


In [ ]:
from scripts.kaggle_final_runtime import export_results
artifact = export_results(ROOT,Path('/kaggle/working/constitution-final-results.zip'))
print(json.dumps(artifact,ensure_ascii=False,indent=2))
print('Download this ZIP from the Kaggle Output panel before the session ends.')


The result ZIP includes recovery lineage, the reused BM25 and hybrid rows, and the new full-text rows. If the first full-text question still runs out of GPU memory, the notebook stops after saving that error; export the partial ZIP and use a GPU with more memory. Successful rows are saved after each question and resume in the same Kaggle working directory. The recovery changes only the memory execution of the full-text system; it is a documented post hoc infrastructure repair, not a new blind test.
